# HR&HRV Grand Final Production Training (45/5)

이 노트북은 `HR&HRV_LOSO.ipynb`의 함수/클래스를 재사용하여,
50명 전체 피험자를 `Train 45 / Val 5`로 분할해 최종 통합 모델을 학습하고 저장합니다.

- 기본 정규화: `subject_zscore`
- 기본 손실함수: `focal`
- 저장 경로: `dataset_root.parent / Save_model / HR_HRV_Grand_Final_Production_Model.pt`

In [1]:
import argparse
import json
import random
from dataclasses import asdict
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import torch


def import_symbols_from_notebook(nb_path: Path, symbols: List[str]) -> Dict[str, object]:
    with nb_path.open("r", encoding="utf-8") as f:
        raw_nb = json.load(f)
    code_cells = [c for c in raw_nb.get("cells", []) if c.get("cell_type") == "code"]
    if len(code_cells) == 0:
        raise RuntimeError(f"No code cells found in {nb_path}")

    namespace: Dict[str, object] = {}
    # HR&HRV_LOSO의 핵심 정의는 1번 코드 셀에 있으므로 그대로 실행해 재사용한다.
    first_source = code_cells[0].get("source", [])
    if isinstance(first_source, list):
        first_source = "".join(first_source)
    exec(first_source, namespace)

    missing = [name for name in symbols if name not in namespace]
    if missing:
        raise RuntimeError(f"Missing symbols from {nb_path.name}: {missing}")

    return {name: namespace[name] for name in symbols}


def build_arg_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description="Train final HR&HRV production model with 45/5 subject split")
    parser.add_argument("--epochs", type=int, default=50)
    parser.add_argument("--batch_size", type=int, default=64)
    parser.add_argument("--lr", type=float, default=0.001)
    parser.add_argument("--hrv_window", type=int, default=30)
    parser.add_argument("--norm_mode", type=str, default="subject_zscore", choices=["subject_zscore", "baseline_zscore", "none"])
    parser.add_argument("--loss_name", type=str, default="focal", choices=["focal", "weighted_bce"])
    parser.add_argument("--seed", type=int, default=42)
    return parser


def main(cli_args: Optional[List[str]] = None) -> Dict[str, object]:
    parser = build_arg_parser()
    if cli_args is None:
        args, _ = parser.parse_known_args()
    else:
        args = parser.parse_args(cli_args)

    notebook_dir = Path.cwd()
    loso_nb_path = notebook_dir / "HR&HRV_LOSO.ipynb"
    if not loso_nb_path.exists():
        raise FileNotFoundError(f"Cannot find source notebook: {loso_nb_path}")

    symbols = import_symbols_from_notebook(
        loso_nb_path,
        symbols=[
            "HyperParameters",
            "find_dataset_root",
            "discover_complete_subjects",
            "build_split_arrays",
            "make_loader",
            "HRHRV1DCNNTransformer",
            "make_criterion",
            "train_one_epoch",
            "evaluate",
        ],
    )

    HyperParameters = symbols["HyperParameters"]
    find_dataset_root = symbols["find_dataset_root"]
    discover_complete_subjects = symbols["discover_complete_subjects"]
    build_split_arrays = symbols["build_split_arrays"]
    make_loader = symbols["make_loader"]
    HRHRV1DCNNTransformer = symbols["HRHRV1DCNNTransformer"]
    make_criterion = symbols["make_criterion"]
    train_one_epoch = symbols["train_one_epoch"]
    evaluate = symbols["evaluate"]

    np.random.seed(args.seed)
    random.seed(args.seed)
    torch.manual_seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(args.seed)

    dataset_root = find_dataset_root()
    all_subjects = discover_complete_subjects(dataset_root)
    if len(all_subjects) < 50:
        raise ValueError(f"Need 50 complete subjects, found {len(all_subjects)}")

    # 요구사항: 전체 50명 기준으로 45/5 split
    all_subjects = sorted(all_subjects, key=lambda x: int(x[3:]))[:50]
    val_subjects = sorted(random.sample(all_subjects, 5), key=lambda x: int(x[3:]))
    train_subjects = sorted([s for s in all_subjects if s not in val_subjects], key=lambda x: int(x[3:]))

    hp = HyperParameters(
        random_seed=args.seed,
        window_size=60,
        stride=10,
        hrv_window=args.hrv_window,
        batch_size=args.batch_size,
        epochs=args.epochs,
        learning_rate=args.lr,
        dropout_rate=0.3,
        weight_decay=1e-4,
        patience=8,
        transformer_heads=8,
        transformer_layers=2,
        transformer_ff_dim=256,
        transformer_dropout=0.1,
        normalization_mode=args.norm_mode,
        loss_name=args.loss_name,
        focal_gamma=2.0,
        focal_alpha=None,
        save_model=False,
        max_folds=None,
    )

    print("=== Grand Final HR&HRV Training (45/5) ===")
    print(f"Normalization: {hp.normalization_mode}")
    print(f"Loss: {hp.loss_name}")
    print(f"HRV window: {hp.hrv_window}")
    print(f"Train subjects ({len(train_subjects)}): {train_subjects}")
    print(f"Val subjects ({len(val_subjects)}): {val_subjects}")

    x_train, y_train = build_split_arrays(
        dataset_root,
        train_subjects,
        hp.window_size,
        hp.stride,
        hp.hrv_window,
        normalization_mode=hp.normalization_mode,
    )
    x_val, y_val = build_split_arrays(
        dataset_root,
        val_subjects,
        hp.window_size,
        hp.stride,
        hp.hrv_window,
        normalization_mode=hp.normalization_mode,
    )

    train_loader = make_loader(x_train, y_train, hp.batch_size, shuffle=True)
    val_loader = make_loader(x_val, y_val, hp.batch_size, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = HRHRV1DCNNTransformer(hp=hp).to(device)
    criterion, loss_cfg = make_criterion(y_train, hp, device)
    optimizer = torch.optim.Adam(model.parameters(), lr=hp.learning_rate, weight_decay=hp.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=4)

    best_val_loss = float("inf")
    best_epoch = -1
    wait = 0
    best_final_state = None
    history: List[Dict[str, float]] = []

    for epoch in range(1, hp.epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_metrics = evaluate(model, val_loader, criterion, device)
        scheduler.step(val_metrics["loss"])

        history.append(
            {
                "epoch": float(epoch),
                "train_loss": float(train_loss),
                "val_loss": float(val_metrics["loss"]),
                "val_accuracy": float(val_metrics["accuracy"]),
                "val_f1": float(val_metrics["f1"]),
                "lr": float(optimizer.param_groups[0]["lr"]),
            }
        )

        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            best_epoch = epoch
            wait = 0
            best_final_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= hp.patience:
                print(f"Early stopping at epoch {epoch} (best epoch: {best_epoch})")
                break

    if best_final_state is None:
        raise RuntimeError("best_final_state is None. Training did not produce a valid checkpoint.")

    final_model_path = dataset_root.parent / "Save_model" / "HR_HRV_Grand_Final_Production_Model.pt"
    final_model_path.parent.mkdir(parents=True, exist_ok=True)

    payload = {
        "model_state_dict": best_final_state,
        "model_name": "HR_HRV_Grand_Final_Production_Model",
        "created_from": "HR&HRV_LOSO.ipynb",
        "best_epoch": int(best_epoch),
        "best_val_loss": float(best_val_loss),
        "device_used_for_training": str(device),
        "hyperparameters": asdict(hp),
        "normalization_mode": hp.normalization_mode,
        "loss_config": loss_cfg,
        "subject_split": {"train": train_subjects, "val": val_subjects},
        "history": history,
    }

    torch.save(payload, final_model_path)

    print(f"Saved final production model to: {final_model_path}")
    return payload


# Notebook 실행 시 기본 인자 사용
final_result = main()
final_result


=== Fold 1/50 | Test: Sub1 ===
Train subjects: 44, Val subjects: 5, Test subjects: 1
Early stopping at epoch 31 (best epoch: 23)
Fold test metrics: {'accuracy': 0.5625, 'f1': 0.72, 'loss': 0.189}

=== Fold 2/50 | Test: Sub2 ===
Train subjects: 44, Val subjects: 5, Test subjects: 1
Early stopping at epoch 26 (best epoch: 18)
Fold test metrics: {'accuracy': 0.9538, 'f1': 0.9703, 'loss': 0.0258}

=== Fold 3/50 | Test: Sub3 ===
Train subjects: 44, Val subjects: 5, Test subjects: 1
Early stopping at epoch 10 (best epoch: 2)
Fold test metrics: {'accuracy': 0.5692, 'f1': 0.7083, 'loss': 0.0573}

=== Fold 4/50 | Test: Sub4 ===
Train subjects: 44, Val subjects: 5, Test subjects: 1
Early stopping at epoch 14 (best epoch: 6)
Fold test metrics: {'accuracy': 0.7692, 'f1': 0.8454, 'loss': 0.0457}

=== Fold 5/50 | Test: Sub5 ===
Train subjects: 44, Val subjects: 5, Test subjects: 1
Early stopping at epoch 10 (best epoch: 2)
Fold test metrics: {'accuracy': 0.3077, 'f1': 0.4706, 'loss': 0.0778}

=== F

{'model_state_dict': {'pos_embedding': tensor([[[-0.0064, -0.0011,  0.0083,  ..., -0.0043,  0.0054, -0.0059],
           [-0.0020,  0.0010,  0.0111,  ..., -0.0010,  0.0042, -0.0065],
           [-0.0041, -0.0015,  0.0045,  ..., -0.0058,  0.0063, -0.0049],
           ...,
           [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
           [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
           [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]]]),
  'cnn.0.weight': tensor([[[ 0.1542,  0.1842, -0.0555,  0.1978, -0.0529],
           [ 0.0632, -0.1294,  0.1731,  0.2543, -0.1780]],
  
          [[ 0.1984,  0.0686,  0.1483,  0.0348,  0.0874],
           [-0.0573,  0.1926,  0.0198, -0.1489,  0.0522]],
  
          [[-0.1153, -0.0089, -0.0937,  0.2204, -0.2106],
           [-0.1519, -0.1027, -0.1903,  0.0026, -0.2936]],
  
          [[ 0.2479, -0.2836,  0.2172,  0.0300, -0.1210],
           [ 0.2008,  0.0659,  0.2625,  0.0566, -0.0611]],
  
          